In [2]:
import yaml

from src.core.ddl_parser import *
from src.jinja.environment import *

In [73]:
test_parser = DDLParser()
file_path = r'C:\Users\dungp\projects\hql_spark_bridge\samples\input\ddl\raw\raw_k2_bank.sql'

with open(file_path, 'r') as f:
    ddl_content = f.read()
    ast = sqlglot.parse(ddl_content, read="hive")

variable_mappings = yaml.load(open(r'C:\Users\dungp\projects\hql_spark_bridge\configs\rules\variable.yaml', 'r'), Loader=yaml.FullLoader)

'raw_schema' in config

True

In [93]:
def map_variable(query):
    for node in query.find_all(exp.Var):
        print(node.this)
        if node.this in variable_mappings:
            # Thay thế node 'Var' bằng một 'Identifier' mới
            node.replace(exp.Identifier(this=variable_mappings[node.this], quoted=False))

In [64]:
# def map_special_value(query):
#     # Thay thế các giá trị đặc biệt
#     for key, value in variable_mappings.items():
#         placeholder = f"${{{key}}}" # Tạo lại placeholder ví dụ: ${batch_timestamp}
#         if placeholder in query:
#             query = query.replace(placeholder, value)

In [97]:
for query in ast:
    map_variable(query)
    spark_sql_query = query.sql(dialect="spark")
    # map_special_value(spark_sql_query)
    print(spark_sql_query)

/* Purpose:    RAW-DDL-CREATE TABLE */ /* Author:     zjj */ /* Usage:      python $ETL_HOME/script/init.py raw k2_bank */ /* CreateDate: 20230907 */ /* FileType:   DDL */ /* Logs: */ /*     1.for hive 3.x on cdp 7.1.5 */ /* 1.0 drop table if exists table */ DROP TABLE IF EXISTS ${}.k2_bank
parquet
/* 1.1 create table */ CREATE TABLE ${}.k2_bank (bankid STRING COMMENT '', localbankcode STRING COMMENT '', bankname STRING COMMENT '', swiftcode STRING COMMENT '', oribankid STRING COMMENT '', approveuser BIGINT COMMENT '', approvets TIMESTAMP COMMENT '', createuser BIGINT COMMENT '', createts TIMESTAMP COMMENT '', updateuser BIGINT COMMENT '', updatets TIMESTAMP COMMENT '', recstatus STRING COMMENT '', etl_timestamp STRING COMMENT 'ETL_processing_time', etl_dt STRING COMMENT 'partition by day') COMMENT '' PARTITIONED BY (etl_dt) USING PARQUET TBLPROPERTIES ('parquet.compression'='SNAPPY', 'external.table.purge'='true')


In [53]:
# samples/converted/ddl/raw

render_template('pyspark/pyspark_basic.jinja', context={})

AttributeError: 'list' object has no attribute 'sql'